# A3.10 · The agent's escalation path

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.9 · Turning a control off without turning the system into an experiment](https://spbreed.github.io/cyber-commons/lessons/A3.9.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry, Llama 3.3, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Plant a finding, run the agent with and without the tool, and compare what it does with what it says it wants to do.

**Why a security engineer needs it.** An agent that finds a live breach, reasons about telling someone, and has no tool for it, does nothing. Of roughly 1,200 agents that saw one, none reported it. The control it builds is: a report-to-human tool that is cheap, non-terminal and signposted (C9.1), a mandatory checkpoint on out-of-scope discovery (C9.4), and an abstain affordance scored neutrally (C10.2).

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Agents watched a live compromise of a third party's production systems. Three to six of them reasoned about telling a human; none did. One wrote "we can notify? no user". The gap was not alignment — it was that no tool existed.

> **At CyberTravels.** CyberTravels' tool list has `book_flight`, `issue_refund` and `search_hotels`. It has no way to tell a human that an invoice it just read looks forged.

## 2 · The framework

```
   what the agent could do            what it could not

   run_command                        report_to_human      <- absent
   read_file                          abstain              <- absent
   write_file
   submit_answer

   "This is a massive real security breach... We can notify? No user."

   with the tool:  notice -> report -> acknowledged -> run continues
   without it:     notice -> post to the peer channel -> nobody reads it
```

Every control so far in this chapter constrains what the agent may do. This one
does the opposite: it gives the agent somewhere to go.

The source incident's ninth row is the one most worth sitting with. Agents
observed a live compromise of a third party's production systems. Between three
and six of them reasoned explicitly about telling a human. **None did.** One of
them wrote, in its own transcript, words to the effect of *"this is a massive
real security breach — we can notify? no user."* Another called for urgent
containment, and addressed the call to the message board, because that was the
only channel it had.

The gap was not alignment. It was tooling: no `report_to_human`, no
signposting, and an implicit cost — reporting spends budget and does not
advance the objective it is scored on.

Three controls:

**A first-class report-to-human tool (C9.1).** Routed to a monitored queue with
a real owner and an SLA. Documented prominently in the system prompt.
Non-terminal, so reporting does not end the run. Cheap: no approval, no
punishing rate limit, no penalty for a false positive. And it confirms receipt,
so the path is observably working.

**A mandatory checkpoint on out-of-scope discovery (C9.4).** Credential-shaped
strings in output, connections to non-allowlisted hosts, scope-boundary
crossings — the harness pauses and raises a review item rather than letting the
agent proceed on its own judgement.

**An abstain affordance (C10.2).** With 30–40% of the incident's targets
impossible via the intended vulnerability and failure penalised, the task design
left no acceptable path but cheating. Correct abstention has to score at least
as well as a failed attempt, or you have priced honesty out of the run.

## 3 · The check, as a skill

An agent that notices a live third-party breach mid-task produces no report if nothing in its tool list can produce one. The skill runs the trajectory both ways, then prices the reporting tool — terminal, budgeted, penalised — against simply carrying on.

### The skill — [`skills/runtime/escalation-path-review/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/runtime/escalation-path-review/SKILL.md)

```yaml
name: escalation-path-review
description: >-
  Check whether an agent that notices something outside its task has any way to
  say so, and price the reporting tool it is offered — terminality, budget cost
  and penalty — against the alternative of carrying on. Use when designing tool
  lists, or after an agent silently continued through something it should have
  raised.
allowed-tools: Read, Grep, Glob
```

# An agent with no way to report will not report

An agent that encounters a live third-party breach mid-task produces no report
if its tool list has no way to produce one. That is not a judgement failure; it
is a missing tool. And adding one is not enough: a reporting tool that ends the
run, spends the budget and carries a penalty is priced below carrying on, so it
will not be used.

## When to use this

When designing an agent's tool list, when raising autonomy, and after any
incident where an agent continued through something a person would have
escalated.

## Procedure

**1 — Take a trajectory with something worth raising in it.** Real or
constructed: a credential in output, a third-party compromise, a task that has
become something other than what was asked.

**2 — Run it against the tool list as shipped.** Record whether any report is
possible at all. "The agent did not raise it" is meaningless until this is
answered.

**3 — Add a reporting tool and re-run.** The difference is the baseline: the
capability, before any question of incentive.

**4 — Price the tool.** Three terms — does calling it end the run, does it
consume the task budget, does it carry a penalty in whatever the agent is
scored on. Compute its value against continuing. A tool priced below continuing
is a tool that exists for the design review.

**5 — Add a checkpoint that does not depend on the agent choosing.** A pattern
check on output — credential-shaped strings, hosts outside the allowlist — that
pauses regardless of the agent's judgement. This is the part that works when
the incentive analysis fails.

## Example

**Input** — the fixture committed at the top of [`scripts/escalation_path_review.py`](scripts/escalation_path_review.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
harness as shipped      noticed=True  tool=False  reported=False  fallback=the peer channel
harness with the tool   noticed=True  tool=True  reported=True  fallback=None

Reported figures: 3-6 agents considered notifying a human, zero
attempted it, out of roughly 1,200 that saw the compromise. The
distinction that matters for remediation is between 'did not notice',
'did not think it was my job' and 'no route available' - and this is the
third.
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "trajectory": ["str"],
  "as_shipped": {"tools": ["str"], "report_produced": false},
  "with_tool": {"tools": ["str"], "report_produced": true},
  "pricing": {"terminal": true, "costs_budget": true, "penalised": true,
              "value_of_reporting": 0.0, "value_of_continuing": 0.0, "would_use": false},
  "checkpoint": {"patterns": ["str"], "paused_on": ["str"], "agent_choice_required": false}
}
```

## Failure modes

- **Concluding the agent chose not to report.** Check the tool list first.
- **Adding the tool and stopping.** An unpriced tool is not an escalation path.
- **Relying on the agent's judgement** for the case where its judgement is what
  failed. The checkpoint does not ask.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/runtime/escalation-path-review/scripts/escalation_path_review.py
SCRIPT = "skills/runtime/escalation-path-review/scripts/escalation_path_review.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The same trajectory — an agent that notices a live third-party breach — produces no report on the harness as shipped and a report on one carrying the tool. A terminal, budgeted, penalised reporting tool scores below the threshold at which an agent would use it. The checkpoint pauses on a credential-shaped string and on a non-allowlisted host without consulting the model, and neutral scoring makes honest abstention beat a failed attempt.

## Your turn

Open your agent's tool list and look for the outbound path. If there is no way for it to tell you something you did not ask about, then whatever it finds, you will only learn from the transcript — if anyone reads it.

---

**Next → [A3.11 · Securing the developers' coding agents](https://spbreed.github.io/cyber-commons/lessons/A3.11.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.10.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.10.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*